In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# Milestone - 2

Q 1) **Hugging Face datasets se load + combined_text + index 51 length**

In [5]:
from datasets import load_dataset

dataset = load_dataset('csv', data_files='/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')

def add_combined_text(example):
    example['combined_text'] = str(example['prompt']) + ' ' + str(example['A'])
    return example

dataset = dataset.map(add_combined_text)

row_51 = dataset['train'][51]
print("combined_text:", row_51['combined_text'])
print("Answer Q1 — Length:", len(row_51['combined_text']))

combined_text: Determine the correct option: What is the reason behind the designation of Class L dwarfs, and what is their color and composition? among the listed options. Class L dwarfs are hotter than M stars and are designated L because L is the remaining letter alphabetically closest to M. They are bright blue in color and are brightest in ultraviolet. Their atmosphere is hot enough to allow metal hydrides and alkali metals to be prominent in their spectra. Some of these objects have masses large enough to support hydrogen fusion and are therefore stars, but most are of substellar mass and are therefore brown dwarfs.
Answer Q1 — Length: 614


Q 2) **BERT tokenizer vocabulary size**  

In [6]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
print("Answer Q2 — Vocab size:", tokenizer.vocab_size)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Answer Q2 — Vocab size: 30522


Q 3) **[SEP] token ID**

In [8]:
sep_id = tokenizer.convert_tokens_to_ids('[SEP]')
print("Answer Q3 — [SEP] token ID:", sep_id)

Answer Q3 — [SEP] token ID: 102


Q 4) **Shape of input_ids tensor**

In [13]:
from transformers import AutoTokenizer
import torch


tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")


prompts = list(dataset["train"]["prompt"])


encoded = tokenizer(
    prompts,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)


print("Shape of input_ids:", encoded["input_ids"].shape)

Shape of input_ids: torch.Size([2000, 128])


Q 5) **Each attention head dimensionality**

In [14]:
hidden_size   = 768
num_heads     = 12
head_dim      = hidden_size // num_heads

print("Answer Q5 — Each attention head size:", head_dim)

Answer Q5 — Each attention head size: 64


Q 6) **last_hidden_state shape for row ID 0**

In [15]:
from transformers import AutoModel

model_bert = AutoModel.from_pretrained('bert-base-uncased')
model_bert.eval()

row0_prompt = dataset['train'][0]['prompt']

inputs = tokenizer(row0_prompt, return_tensors='pt')

import torch
with torch.no_grad():
    outputs = model_bert(**inputs)

print("Answer Q6 — last_hidden_state shape:", outputs.last_hidden_state.shape)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Answer Q6 — last_hidden_state shape: torch.Size([1, 31, 768])


Q 7) **Sum of first 5 float values of [CLS] vector**

In [16]:
cls_vector = outputs.last_hidden_state[0][0]

first_5 = cls_vector[:5].tolist()
print("First 5 values:", first_5)
print("Answer Q7 — Sum of first 5:", round(sum(first_5), 4))

First 5 values: [-0.46766412258148193, -0.0754447653889656, -0.20190055668354034, -0.00706420186907053, -0.4480222761631012]
Answer Q7 — Sum of first 5: -1.2001


Q 8) **Attention weight [CLS] pays to "fusion"**

In [17]:
model_attn = AutoModel.from_pretrained('bert-base-uncased', output_attentions=True)
model_attn.eval()

sentence = "Light-ion fusion is a technique."
inputs_attn = tokenizer(sentence, return_tensors='pt')

with torch.no_grad():
    outputs_attn = model_attn(**inputs_attn)

# Find token index of "fusion"
tokens = tokenizer.convert_ids_to_tokens(inputs_attn['input_ids'][0])
print("Tokens:", tokens)
fusion_idx = tokens.index('fusion')
print("Fusion index:", fusion_idx)

# Last layer, first head, CLS token (index 0) attention to fusion
attn_weight = outputs_attn.attentions[-1][0][0][0][fusion_idx].item()
print("Answer Q8 — Attention weight:", round(attn_weight, 4))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Tokens: ['[CLS]', 'light', '-', 'ion', 'fusion', 'is', 'a', 'technique', '.', '[SEP]']
Fusion index: 4
Answer Q8 — Attention weight: 0.1025


Q 9) **Cosine similarity prompt vs Option B for row ID 0**

In [18]:
from sentence_transformers import SentenceTransformer, util

sbert = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

row0 = dataset['train'][0]

prompt_emb  = sbert.encode(str(row0['prompt']), convert_to_tensor=True)
optionB_emb = sbert.encode(str(row0['B']),      convert_to_tensor=True)

sim = util.cos_sim(prompt_emb, optionB_emb).item()
print("Answer Q9 — Cosine similarity:", round(sim, 4))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Answer Q9 — Cosine similarity: 0.7658


Q 10) **MAP@3 of MiniLM pipeline + count where TF-IDF fails but MiniLM succeeds**

In [19]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity as sk_cosine
from tqdm import tqdm

train_pd    = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
OPTION_COLS = ['A', 'B', 'C', 'D', 'E']

def map_at_3(ground_truth, predictions):
    score = 0.0
    for k, pred in enumerate(predictions[:3], start=1):
        if pred == ground_truth:
            score = 1.0 / k
            break
    return score

# TF-IDF pipeline
combined = []
for _, row in train_pd.iterrows():
    parts = [str(row['prompt'])] + [str(row[c]) for c in OPTION_COLS]
    combined.append(' '.join(parts))

vectorizer = TfidfVectorizer(stop_words='english')
vectorizer.fit(combined)

tfidf_top3 = []
for _, row in train_pd.iterrows():
    prompt_vec = vectorizer.transform([str(row['prompt'])])
    sims = {}
    for opt in OPTION_COLS:
        opt_vec   = vectorizer.transform([str(row[opt])])
        sims[opt] = sk_cosine(prompt_vec, opt_vec)[0][0]
    ranked = sorted(sims, key=sims.get, reverse=True)[:3]
    tfidf_top3.append(ranked)

# MiniLM pipeline
miniLM_top3 = []
for _, row in tqdm(train_pd.iterrows(), total=len(train_pd), desc="MiniLM"):
    prompt_emb = sbert.encode(str(row['prompt']), convert_to_tensor=True)
    sims = {}
    for opt in OPTION_COLS:
        opt_emb   = sbert.encode(str(row[opt]), convert_to_tensor=True)
        sims[opt] = util.cos_sim(prompt_emb, opt_emb).item()
    ranked = sorted(sims, key=sims.get, reverse=True)[:3]
    miniLM_top3.append(ranked)

# MAP@3 of MiniLM
minilm_scores = [map_at_3(true, pred) for true, pred in zip(train_pd['answer'], miniLM_top3)]
print("Answer Q10 — MiniLM MAP@3:", round(np.mean(minilm_scores), 4))

# Count where TF-IDF fails but MiniLM succeeds
count = 0
for i, row in train_pd.iterrows():
    ans = row['answer']
    if ans not in tfidf_top3[i] and ans in miniLM_top3[i]:
        count += 1

print("Answer Q10 — Count:", count)

MiniLM: 100%|██████████| 2000/2000 [03:10<00:00, 10.50it/s]

Answer Q10 — MiniLM MAP@3: 0.4231
Answer Q10 — Count: 502


Q11) **Zero-shot classification, top-ranked probability**

In [20]:
from transformers import pipeline

classifier = pipeline('zero-shot-classification', model='facebook/bart-large-mnli')

row1 = dataset['train'][1]
prompt_text     = str(row1['prompt'])
candidate_labels = [str(row1['A']), str(row1['B']), str(row1['C'])]

result = classifier(prompt_text, candidate_labels)
print("Labels :", result['labels'])
print("Scores :", result['scores'])
print("Answer Q11 — Top score:", round(result['scores'][0], 4))

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Labels : ['Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively easy to implement and can be done in an efficient manner, requiring only a vacuum tube, a pair of electrodes, and a high-voltage transformer. Fusion can be observed with as little as 10 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively difficult to implement and requires a complex system of vacuum tubes, electrodes, and transformers. Fusion can be observed with as little as 100 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce heavy-ion fusion reactions. This method is relatively difficult to imp

Q12) **Multi_label=True, absolute difference of sum**

In [22]:
result_multi = classifier(prompt_text, candidate_labels, multi_label=True)
print("Multi-label scores:", result_multi['scores'])

sum_softmax = sum(result['scores'])
sum_sigmoid = sum(result_multi['scores'])

diff = abs(sum_softmax - sum_sigmoid)
print(f"Sum softmax : {round(sum_softmax, 4)}")
print(f"Sum sigmoid : {round(sum_sigmoid, 4)}")
print(f"Answer Q12 — Absolute difference: {round(diff, 4)}")

Multi-label scores: [0.00046927202492952347, 2.063589454337489e-05, 1.9700619304785505e-05]
Sum softmax : 1.0
Sum sigmoid : 0.0005
Answer Q12 — Absolute difference: 0.9995


Q13) **Flan-T5 small generative QA**

In [25]:
from transformers import pipeline

flan = pipeline(
    task="text-generation",
    model="google/flan-t5-small"
)

row0 = dataset["train"][0]

input_text = (
    f"Question: {row0['prompt']}. "
    f"Is the correct answer A: {row0['A']} or B: {row0['B']}? "
    "Answer with just the letter A or B."
)

output = flan(input_text, max_new_tokens=5)

print(output[0]["generated_text"])

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM', 'Ernie4_5ForCausalLM', 'Ernie4_5_MoeForCausalLM', 'Exaone4ForCausalLM', 'FalconForCausalLM', 'FalconH1ForCausalLM', 'FalconMambaForCausa

Question: Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.. Is the correct answer A: Martin Heidegger believes that humans exist within a time continuum that is infinite and does not have a defined beginning or end. The relationship to the past involves acknowledging it as a historical era, and the relationship to the future involves creating a world that will endure beyond one's own time. or B: Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement.? Answer with just the letter A or B.


In [26]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

row0 = dataset["train"][0]

input_text = (
    f"Question: {row0['prompt']}. "
    f"Is the correct answer A: {row0['A']} or B: {row0['B']}? "
    "Answer with just the letter A or B."
)

inputs = tokenizer(input_text, return_tensors="pt")

outputs = model.generate(**inputs, max_new_tokens=5)

answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("Answer Q13 — Output:", repr(answer))

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Answer Q13 — Output: 'B'
